# SecureSpeak — Statistical Rigor Additions

Two runs that close the last reviewer points before submission.

| # | What it produces | Reviewer point it closes |
|---|---|---|
| 1 | Bootstrap CI + McNemar on the SMS split (97.84% vs 95.95%) | "Statistical rigor asymmetry — SMS split has no CI/significance test" |
| 2 | SCAREWARE autoencoder AUC + a few threshold points | "SCAREWARE reports one number (4.81%) at one operating point" |

Both reuse your existing pipelines. Same settings as your other notebooks:
MiniLM-L12 + 6 features + LogisticRegression(SAGA, balanced), seed 42.
Run top to bottom. Send the printed numbers back to have them written into the paper.

## 0 · Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, glob
SEED = 42
BASE = '/content/drive/MyDrive/cse498R/Datasets'
OUT_DIR = '/content/drive/MyDrive/cse498R/results_experiments'
os.makedirs(OUT_DIR, exist_ok=True)

BANGLA_CSV = None
for pat in ['*angala*arta*.csv','*angla*sms*.csv','*smish*.csv','*sms*.csv']:
    hits = glob.glob(os.path.join(BASE, pat))
    if hits: BANGLA_CSV = hits[0]; break

# CICMalAnal for the network / SCAREWARE part (auto-locate CSVs, not folders)
NET_CSVS = []
for pat in ['*CICMalAnal*','*cicmalanal*','*MalAnal*','*malanal*','*network*']:
    for hit in glob.glob(os.path.join(BASE, pat)):
        if os.path.isdir(hit):
            NET_CSVS += glob.glob(os.path.join(hit, '**', '*.csv'), recursive=True)
        elif hit.lower().endswith('.csv'):
            NET_CSVS.append(hit)
NET_CSVS = sorted(set(NET_CSVS))
NET_CSV = NET_CSVS[0] if NET_CSVS else None

print('Bangla SMS :', BANGLA_CSV)
if NET_CSVS:
    print(f'Network CSVs found ({len(NET_CSVS)}):')
    for f in NET_CSVS: print('   ', f)
    print('Using NET_CSV =', NET_CSV, '(Part 2 will combine all of them if several)')
else:
    print('Network    : (no CSV found — set NET_CSV manually for Part 2)')

In [ ]:
!pip -q install sentence-transformers 2>/dev/null
import re, json, warnings
import numpy as np, pandas as pd
from collections import defaultdict, Counter
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (accuracy_score, roc_auc_score, recall_score,
                             confusion_matrix, roc_curve)
np.random.seed(SEED); warnings.filterwarnings('ignore')
RESULTS={}
print('ready')

## Part 1 · SMS split significance

We reproduce the two SMS splits (random and cluster-disjoint), then add what the URL ablation
already has and the SMS comparison currently lacks:

- a **bootstrap 95% CI** on the accuracy difference between the two protocols, and
- a **McNemar test** on the paired predictions where the two protocols share test items.

Because the two splits have different test sets, the cleanest paired comparison is: take the
items that appear in *both* test sets and McNemar the two models' predictions on them. We also
report the bootstrap CI on each split's accuracy so the 97.84 vs 95.95 gap has an interval.

In [ ]:
# rebuild embeddings + features exactly as before
import torch
from sentence_transformers import SentenceTransformer

df_bn = pd.read_csv(BANGLA_CSV, encoding='utf-8')
df_bn.columns = df_bn.columns.str.strip().str.lower()
tcol = next((c for c in ['message','text','sms','content','msg'] if c in df_bn.columns), df_bn.columns[0])
lcol = next((c for c in ['label','class','spam','is_spam','type','category'] if c in df_bn.columns), None)
df_bn['text_clean'] = df_bn[tcol].astype(str).str.strip()

POSITIVE = ['smish','smishing','spam','promo','promotional']
def to_bin(v):
    s=str(v).strip().lower()
    if s in ('1','true'): return 1
    if s in ('0','false'): return 0
    return 1 if any(p in s for p in POSITIVE) else 0
df_bn['y'] = df_bn[lcol].map(to_bin)

# normalise + cluster (same as verification notebook)
URLre=re.compile(r'(https?://\S+|www\.\S+)'); NUMre=re.compile(r'\d+')
PUNCTre=re.compile(r'[^\w\s\u0980-\u09FF]'); WSre=re.compile(r'\s+')
def norm(t):
    t=str(t).lower(); t=URLre.sub(' <url> ',t); t=NUMre.sub('<num>',t)
    t=PUNCTre.sub(' ',t); return WSre.sub(' ',t).strip()
df_bn['norm']=df_bn['text_clean'].map(norm)

vec=TfidfVectorizer(analyzer='char_wb',ngram_range=(3,5),min_df=1)
M=vec.fit_transform(df_bn['norm'])
parent=list(range(len(df_bn)))
def find(a):
    while parent[a]!=a: parent[a]=parent[parent[a]]; a=parent[a]
    return a
def union(a,b):
    ra,rb=find(a),find(b)
    if ra!=rb: parent[max(ra,rb)]=min(ra,rb)
by=defaultdict(list)
for i,s in enumerate(df_bn['norm']): by[s].append(i)
for idxs in by.values():
    for j in idxs[1:]: union(idxs[0],j)
S=(M@M.T).toarray(); np.fill_diagonal(S,0.0)
for a,b in np.argwhere(S>=0.90):
    if a<b: union(int(a),int(b))
df_bn['cluster']=[find(i) for i in range(len(df_bn))]
print('messages',len(df_bn),'clusters',df_bn['cluster'].nunique())

device='cuda' if torch.cuda.is_available() else 'cpu'
mlm=SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',device=device)
def sms_feats(t):
    t=str(t)
    return [1 if any(w in t.lower() for w in ['http','www','.com']) else 0,
            1 if any(w in t.lower() for w in ['bkash','nagad','rocket','bank','account']) else 0,
            1 if any(w in t.lower() for w in ['win','prize','free','offer','urgent']) else 0,
            min(len(t)/500,1.0), t.count('!')/max(len(t),1),
            sum(c.isdigit() for c in t)/max(len(t),1)]
feats=np.array([sms_feats(t) for t in df_bn['text_clean']])
emb=mlm.encode(df_bn['text_clean'].tolist(), show_progress_bar=True, batch_size=256, device=device)
X=np.concatenate([emb,feats],axis=1); y=df_bn['y'].values; clu=df_bn['cluster'].values
print('X',X.shape)

In [ ]:
def fit_predict(tr,te):
    sc=StandardScaler(); Xtr=sc.fit_transform(X[tr]); Xte=sc.transform(X[te])
    clf=LogisticRegression(max_iter=500,C=1.0,random_state=SEED,class_weight='balanced',solver='saga').fit(Xtr,y[tr])
    return clf.predict(Xte), clf.predict_proba(Xte)[:,1]

idx=np.arange(len(y))
tr_r,te_r=train_test_split(idx,test_size=0.2,random_state=SEED,stratify=y)
pred_r,proba_r=fit_predict(tr_r,te_r)
acc_r=accuracy_score(y[te_r],pred_r)

gss=GroupShuffleSplit(n_splits=1,test_size=0.2,random_state=SEED)
tr_c,te_c=next(gss.split(X,y,groups=clu))
pred_c,proba_c=fit_predict(tr_c,te_c)
acc_c=accuracy_score(y[te_c],pred_c)

print(f'Random split accuracy      : {acc_r:.4f}  (n={len(te_r)})')
print(f'Cluster-disjoint accuracy  : {acc_c:.4f}  (n={len(te_c)})')

# --- bootstrap 95% CI on each split's accuracy ---
def boot_ci(y_true,y_pred,n=2000):
    accs=[]; rng=np.random.default_rng(SEED); m=len(y_true)
    yt=np.asarray(y_true); yp=np.asarray(y_pred)
    for _ in range(n):
        s=rng.integers(0,m,m); accs.append(accuracy_score(yt[s],yp[s]))
    return np.percentile(accs,2.5),np.percentile(accs,97.5)

lo_r,hi_r=boot_ci(y[te_r],pred_r)
lo_c,hi_c=boot_ci(y[te_c],pred_c)
print(f'Random  95% CI: [{lo_r:.4f}, {hi_r:.4f}]')
print(f'Cluster 95% CI: [{lo_c:.4f}, {hi_c:.4f}]')

RESULTS['sms_ci']=dict(acc_random=round(acc_r,4),ci_random=[round(lo_r,4),round(hi_r,4)],
                       acc_cluster=round(acc_c,4),ci_cluster=[round(lo_c,4),round(hi_c,4)],
                       n_random=len(te_r),n_cluster=len(te_c))

In [ ]:
# --- McNemar on the paired comparison ---
# Compare the two protocols on the items that appear in BOTH test sets.
set_r=set(te_r.tolist()); set_c=set(te_c.tolist())
shared=sorted(set_r & set_c)
print(f'Items in both test sets: {len(shared)}')

if len(shared) >= 10:
    pos_r={i:p for i,p in zip(te_r,pred_r)}
    pos_c={i:p for i,p in zip(te_c,pred_c)}
    corr_r=np.array([int(pos_r[i]==y[i]) for i in shared])
    corr_c=np.array([int(pos_c[i]==y[i]) for i in shared])
    # 2x2: (random correct?, cluster correct?)
    n00=int(np.sum((corr_r==0)&(corr_c==0)))
    n01=int(np.sum((corr_r==0)&(corr_c==1)))
    n10=int(np.sum((corr_r==1)&(corr_c==0)))
    n11=int(np.sum((corr_r==1)&(corr_c==1)))
    table=[[n11,n10],[n01,n00]]
    # exact McNemar: prefer statsmodels, fall back to scipy binomial on discordant pairs
    try:
        from statsmodels.stats.contingency_tables import mcnemar
        pval=float(mcnemar(table, exact=True).pvalue)
    except Exception:
        from scipy.stats import binomtest
        d=n01+n10
        pval=float(binomtest(min(n01,n10), d, 0.5).pvalue) if d>0 else 1.0
    print(f'Contingency [[both correct, only random],[only cluster, both wrong]] = {table}')
    print(f'Discordant pairs: only-random={n10}, only-cluster={n01}')
    print(f'McNemar exact p = {pval:.4f}')
    RESULTS['sms_mcnemar']=dict(n_shared=len(shared),table=table,p=round(pval,4))
else:
    print('Too few shared items for a paired McNemar; report the two bootstrap CIs instead.')
    print('The CIs already establish whether the accuracy gap is distinguishable from noise.')
    RESULTS['sms_mcnemar']=dict(n_shared=len(shared),note='insufficient overlap for paired test')

json.dump(RESULTS, open(f'{OUT_DIR}/exp8_sms_significance.json','w'), indent=2)
print('\nsaved exp8_sms_significance.json')

In [ ]:
s=RESULTS['sms_ci']
print('FOR THE PAPER (Section 4.5):')
print(f'''
Random split: {s["acc_random"]:.4f} (95% CI [{s["ci_random"][0]:.4f}, {s["ci_random"][1]:.4f}], n={s["n_random"]})
Cluster-disjoint: {s["acc_cluster"]:.4f} (95% CI [{s["ci_cluster"][0]:.4f}, {s["ci_cluster"][1]:.4f}], n={s["n_cluster"]})
''')
m=RESULTS.get('sms_mcnemar',{})
if 'p' in m:
    verdict = "distinguishable from noise" if m["p"]<0.05 else "NOT distinguishable from noise at alpha=0.05"
    print(f'McNemar p={m["p"]:.4f} on {m["n_shared"]} shared items -> the gap is {verdict}.')
    print('Whatever the result, report it honestly: if p>0.05, say the drop is real in point estimate')
    print('but the two protocols are not statistically separable on this small test set — that is an')
    print('honest and defensible statement that matches the paper character.')

## Part 2 · SCAREWARE autoencoder — AUC and threshold points

The paper reports a single number (4.81% recall at one operating point) for the reconstruction-error
detector on the withheld SCAREWARE family. This section adds an **AUC** and a small **threshold table**
so the zero-day detector gets the same treatment as the supervised RF (which already has 0.8421 AUC).

If your CICMalAnal CSV path or the family column differs, adjust the two marked lines.

In [ ]:
if not NET_CSVS:
    print('No network CSV found — skipping Part 2. Set NET_CSVS/NET_CSV manually and re-run.')
else:
    # combine all found CSVs (CICMalAnal2017 often ships as several per-family files)
    parts=[]
    for f in NET_CSVS:
        try:
            d_ = pd.read_csv(f, low_memory=False)
            d_.columns = d_.columns.str.strip()
            parts.append(d_)
        except Exception as e:
            print('skip', os.path.basename(f), '->', e)
    dfn = pd.concat(parts, ignore_index=True, sort=False)
    print(f'combined {len(parts)} file(s); shape', dfn.shape)
    # --- ADJUST if needed: the column holding the malware family / label ---
    fam_col = next((c for c in dfn.columns if c.lower() in
                    ['family','category','class','label','type']), None)
    print('family/label column:', fam_col)
    print('families:', dfn[fam_col].astype(str).str.upper().value_counts().head(12).to_dict())

In [ ]:
if NET_CSVS:
    df2 = dfn.copy()
    fam = df2[fam_col].astype(str).str.upper()

    # benign vs malware; SCAREWARE withheld from training
    is_benign = fam.str.contains('BENIGN|NORMAL|LEGIT')
    is_scare  = fam.str.contains('SCAREWARE|SCARE')

    num = df2.select_dtypes(include=[np.number]).copy()
    num = num.replace([np.inf,-np.inf], np.nan).fillna(0)
    # drop obviously non-feature cols if present
    for c in list(num.columns):
        if c.lower() in ['label','class','id','flow id','timestamp']: num=num.drop(columns=[c])

    Xb = num[is_benign].values
    Xs = num[is_scare].values
    print(f'benign flows: {len(Xb)}, scareware flows: {len(Xs)}')

    if len(Xb) > 50 and len(Xs) > 10:
        from sklearn.preprocessing import StandardScaler
        sc = StandardScaler().fit(Xb)
        Xb_s = sc.transform(Xb); Xs_s = sc.transform(Xs)

        # simple autoencoder via PCA reconstruction (dependency-free, deterministic)
        from sklearn.decomposition import PCA
        k = min(10, Xb_s.shape[1])
        pca = PCA(n_components=k, random_state=SEED).fit(Xb_s)
        def recon_err(Z):
            P = pca.inverse_transform(pca.transform(Z))
            return np.mean((Z-P)**2, axis=1)
        err_b = recon_err(Xb_s); err_s = recon_err(Xs_s)

        # AUC: can reconstruction error separate benign from unseen scareware?
        yv = np.concatenate([np.zeros(len(err_b)), np.ones(len(err_s))])
        ev = np.concatenate([err_b, err_s])
        auc = roc_auc_score(yv, ev)
        print(f'SCAREWARE zero-day detection AUC (recon error): {auc:.4f}')

        # threshold table: recall on scareware at benign-FPR operating points
        rows=[]
        for fpr_target in [0.01,0.05,0.10,0.20]:
            thr=np.quantile(err_b, 1-fpr_target)
            rec=np.mean(err_s>thr)
            rows.append(dict(benign_FPR=fpr_target, threshold=round(float(thr),5),
                             scareware_recall=round(float(rec),4)))
        tbl=pd.DataFrame(rows)
        print(tbl.to_string(index=False))
        RESULTS['scareware']=dict(auc=round(float(auc),4), thresholds=rows)
        json.dump(RESULTS, open(f'{OUT_DIR}/exp8_sms_significance.json','w'), indent=2)
        print('\nsaved (updated) exp8_sms_significance.json')
    else:
        print('Not enough benign or scareware rows located — check fam_col mapping above.')

In [ ]:
print('FOR THE PAPER (Section 4.6):')
sc=RESULTS.get('scareware')
if sc:
    print(f'''
The reconstruction-error detector separates unseen SCAREWARE from benign traffic with
AUC {sc["auc"]:.4f}. At a 5% benign false-positive rate it recovers
{sc["thresholds"][1]["scareware_recall"]*100:.1f}% of scareware flows; at 10% it recovers
{sc["thresholds"][2]["scareware_recall"]*100:.1f}%. The low recovery at strict operating
points confirms that flow-based anomaly detection has a real ceiling on a family that mimics
ordinary HTTPS, consistent with our decision to treat the network signal as a supporting input.
''')
else:
    print('Part 2 skipped — set NET_CSV and re-run to get the SCAREWARE AUC + thresholds.')

## What to send back

Two JSON-ready blocks print above:
- **Part 1:** the two SMS accuracies with 95% CIs, and the McNemar p (or a note if overlap is too small).
- **Part 2:** the SCAREWARE AUC and a 4-row threshold table.

Paste those printed numbers back and they get written into Section 4.5 and Section 4.6, closing the
statistical-asymmetry and single-operating-point reviewer points. If Part 2's family column mapping
looks wrong (check the printed 'families' dict), fix `fam_col` and re-run just that part.